[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/sh2026-workshop-v1.0.0/workshop/03_contextual_ner.ipynb)

# AI and NLP for Spatial Humanities
## 03 - Contextual NER: generalisation, model ontologies and hybrid enrichment

**Duration:** 50-60 minutes

Notebook 02 showed a transparent deterministic baseline. We now introduce statistical/contextual named-entity recognition and compare three different questions that are often accidentally collapsed into one:

1. **Did the model find the named place?**
2. **Does the model's label inventory match our scholarly ontology?**
3. **How much of the broader spatial-humanities task can NER represent at all?**

> A low score can mean a detection failure, a boundary disagreement, an ontology mismatch, or simply that the model was never trained to express the phenomenon being scored.

### Learning outcomes
You will:
- run spaCy NER without project rules;
- inspect the model's native labels before harmonising them;
- map GPE/LOC/FAC into the benchmark's `TOPONYM` label explicitly;
- compare a pure contextual model with a hybrid model enriched by project resources;
- distinguish **within-task NER accuracy** from **representational reach**;
- optionally run a Hugging Face CoNLL-style NER model;
- save comparable metrics and telemetry for later keynote figures.

In [ ]:
import json, os, pathlib, subprocess, sys, urllib.request

SETUP_URL = (
    "https://raw.githubusercontent.com/IgnatiusEzeani/"
    "spatial-humanities-2026/sh2026-workshop-v1.0.0/workshop/sh2026_setup.py"
)
local_setup = pathlib.Path("workshop/sh2026_setup.py")
setup_path = local_setup if local_setup.exists() else pathlib.Path("/content/sh2026_setup.py")
if not local_setup.exists():
    urllib.request.urlretrieve(SETUP_URL, setup_path)
if str(setup_path.parent.resolve()) not in sys.path:
    sys.path.insert(0, str(setup_path.parent.resolve()))

import sh2026_setup as sh
ctx = sh.setup()
repo_dir = ctx.project
data_dir = ctx.data
output_dir = ctx.outputs
FAST_MODE = ctx.fast_mode

from workshop_support import display, checkpoint, compare_spans, show_fields, show_journey, show_spans


In [ ]:
import pandas as pd
from IPython.display import display
from spatio_textual.gold import assert_valid_gold, load_gold_jsonl, score_span_annotations
from spatio_textual.evaluation import (
    harmonize_ner_entities, label_inventory, reference_spans_for_ner,
    supported_reference_fraction,
)
from spatio_textual.utils import Annotator, load_spacy_model

gold_path=repo_dir/"workshop"/"data"/"gold_reference_v0.1.jsonl"
records=load_gold_jsonl(gold_path); assert_valid_gold(records)
gold={r["example_id"]:r for r in records}
print(f"Loaded {len(records)} validated teaching/development references.")

## 1. First remove the project rules

`spatio-textual` normally enriches spaCy with an `EntityRuler` built from project resources. That is useful in production, but it would make a supposedly 'pure spaCy' comparison partly rule-based.

So we explicitly run:

`add_entity_ruler=False`

for the contextual-only condition. This is a small but important experimental-control decision.

In [ ]:
nlp_pure=load_spacy_model("en_core_web_sm",add_entity_ruler=False)
print("Pure pipeline components:",nlp_pure.pipe_names)
print("Pipeline metadata name:",nlp_pure.meta.get("name"))
if "ner" not in nlp_pure.pipe_names:
    raise RuntimeError("en_core_web_sm did not load with an NER component. Check the notebook installation step.")
pure_ner=Annotator(nlp=nlp_pure,model_name="en_core_web_sm:pure",link_places=False)

## 2. Inspect native model labels before remapping them

The benchmark reference uses `TOPONYM` because its purpose is methodological comparison across systems. spaCy uses labels such as `GPE`, `LOC` and `FAC`.

Scoring those raw strings directly would call a correct `GPE` prediction wrong merely because the human reference says `TOPONYM`. We therefore preserve the native label as `model_label` and map only spatial labels to the common comparison label.

In [ ]:
record=gold["synthetic_qa_journey"]
raw=pure_ner.annotate(record["text"],include_entities=True,include_verbs=False,include_events=False,include_text=True)
print("Native label inventory:",label_inventory(raw["entities"]))
display(pd.DataFrame(raw["entities"])[["text","label","start_char","end_char","source"]])

harmonized=harmonize_ner_entities(raw["entities"])
print("\nHarmonized spatial labels:")
display(pd.DataFrame(harmonized)[["text","model_label","label","start_char","end_char"]])

### Why PERSON/ORG are not counted as spatial false positives

A general NER model is designed to find non-spatial entities too. The spatial benchmark therefore filters those outputs rather than penalising a model for correctly doing work outside the target task.

This is an example of **evaluation design being part of the scholarly method**.

## 3. Fair named-place score

First score only the part of the human ontology that an off-the-shelf NER model is expected to express: named places (`TOPONYM`).

This is our **within-ontology named-place score**.

In [ ]:
reference_places=reference_spans_for_ner(record,include_geonouns=False)
for mode in ("exact","overlap"):
    score=score_span_annotations(harmonized,reference_places,match=mode,label_sensitive=True)
    print(mode,{k:score[k] for k in ("tp","fp","fn","precision","recall","f1")})

## 4. Representational reach is a different question

The human reference also contains geo-nouns, distances, directions, movement cues, transport, relations and sense-of-place descriptions. A conventional place NER ontology cannot express most of these labels even if its named-place recognition is perfect.

We therefore compute an **ontology ceiling** separately from model recall.

In [ ]:
ceiling_rows = []
import matplotlib.pyplot as plt

for rec in records:
    pure_ceiling = supported_reference_fraction(rec, {"TOPONYM"})
    hybrid_ceiling = supported_reference_fraction(rec, {"TOPONYM", "GEONOUN"})
    ceiling_rows.append({
        "example_id": rec["example_id"],
        "Named places only": pure_ceiling["max_span_recall_from_ontology"],
        "Places + geographic nouns": hybrid_ceiling["max_span_recall_from_ontology"],
    })

ceiling_df = pd.DataFrame(ceiling_rows).set_index("example_id")
display(ceiling_df)
ceiling_df.plot.bar(figsize=(8, 4), ylim=(0, 1), ylabel="Maximum recall allowed by ontology", rot=0)
plt.title("The label inventory sets a ceiling before modelling begins")
plt.show()


Do not label the final column `NER recall`. It is not. It is the proportion of our broader annotation ontology that the output vocabulary is capable of representing **in principle**.

## 5. Pure contextual model versus project-enriched hybrid

Now load the normal `spatio-textual` configuration, which adds project resource patterns before spaCy's statistical NER component.

This gives us an honest comparison between:

- **Contextual-only:** `en_core_web_sm`
- **Hybrid:** resource rules + `en_core_web_sm`

The hybrid can recognise domain categories such as `GEONOUN` that the pretrained NER label inventory does not natively provide.

In [ ]:
nlp_hybrid=load_spacy_model("en_core_web_sm",add_entity_ruler=True)
print("Hybrid pipeline components:",nlp_hybrid.pipe_names)
hybrid_ner=Annotator(nlp=nlp_hybrid,model_name="en_core_web_sm+project_resources",link_places=False)

In [ ]:
def evaluate_condition(annotator,rec,method,include_geonouns=False):
    out=annotator.annotate(rec["text"],include_entities=True,include_verbs=False,include_events=False)
    pred=harmonize_ner_entities(out["entities"])
    ref=reference_spans_for_ner(rec,include_geonouns=include_geonouns)
    exact=score_span_annotations(pred,ref,match="exact",label_sensitive=True)
    overlap=score_span_annotations(pred,ref,match="overlap",label_sensitive=True)
    tel=out["telemetry"][0]
    return {
        "example_id":rec["example_id"],
        "method":method,
        "target_ontology":"TOPONYM+GEONOUN" if include_geonouns else "TOPONYM",
        "reference_spans":len(ref),
        "predicted_spatial_spans":len(pred),
        "exact_precision":exact["precision"],
        "exact_recall":exact["recall"],
        "exact_f1":exact["f1"],
        "overlap_f1":overlap["f1"],
        "latency_ms":tel["latency_ms"],
        "cost_usd_est":tel["cost_usd_est"],
        "native_label_inventory":label_inventory(out["entities"]),
    }

rows=[]
for rec in records:
    rows.append(evaluate_condition(pure_ner,rec,"spaCy small: pure NER",False))
    rows.append(evaluate_condition(hybrid_ner,rec,"spaCy small + project resources",True))
comparison_df=pd.DataFrame(rows)
display(comparison_df)

### Be careful interpreting this table

The two rows are not targeting exactly the same ontology:

- pure NER is assessed on `TOPONYM`;
- the hybrid is assessed on `TOPONYM + GEONOUN`.

For a strict head-to-head named-place comparison, evaluate both on `TOPONYM` only. For a methodological comparison, also show how enrichment expands the representational vocabulary.

In [ ]:
head_to_head=[]
for rec in records:
    for annotator,method in [(pure_ner,"spaCy small: pure NER"),(hybrid_ner,"spaCy small + project resources")]:
        out=annotator.annotate(rec["text"],include_entities=True,include_verbs=False,include_events=False)
        pred=[p for p in harmonize_ner_entities(out["entities"]) if p["label"]=="TOPONYM"]
        ref=reference_spans_for_ner(rec,include_geonouns=False)
        score=score_span_annotations(pred,ref,match="exact")
        head_to_head.append({"example_id":rec["example_id"],"method":method,"precision":score["precision"],"recall":score["recall"],"f1":score["f1"]})
display(pd.DataFrame(head_to_head))

## 6. Error analysis on difficult spatial cases

Inspect four different stressors:

1. historical/variant spelling: `Ulleswater`;
2. ambiguous toponym: `Cambridge`;
3. historical polity: `Czechoslovakia`;
4. relational space dominated by common nouns rather than proper names.

In [ ]:
from spatio_textual.rules import RuleGazetteerAnnotator

rule_annotator = RuleGazetteerAnnotator(
    gazetteer_path=ctx.data / "teaching_gazetteer.csv", link_places=False
)
for example_id in [
    "cldw_penrith_pooley_bridge",
    "synthetic_ambiguous_cambridge",
    "synthetic_historical_polity",
    "synthetic_relational_space",
]:
    rec = gold[example_id]
    print()
    print(rec['title'])
    compare_spans(rec["text"], {
        "rules + gazetteer": rule_annotator.annotate(rec["text"])["spans"],
        "spaCy small": harmonize_ner_entities(
            pure_ner.annotate(rec["text"], include_entities=True)["entities"]),
        "spaCy + project resources": harmonize_ner_entities(
            hybrid_ner.annotate(rec["text"], include_entities=True)["entities"]),
        "human reference": rec["spans"],
    })


### Discussion prompts
- Does contextual NER recover names absent from our teaching gazetteer?
- Does it preserve unusual historical spelling?
- Can it tell which `Cambridge` is meant, or only recognise the string as a place?
- What happens when most spatial meaning is encoded by `village`, `woods`, `river`, `nearest`, `beyond`, `left`?
- Which errors are easier to repair: missing dictionary forms or model-domain/ontology mismatches?

## 7. Optional heavy comparison: Hugging Face NER

The repository includes `HFNERAnnotator` for transformer token-classification models. The live route below uses `dslim/bert-base-NER` at an **exact pinned Hugging Face revision**.

For the default CPU-friendly workshop path, we bundle a **real precomputed run** over these same teaching references. The fallback records the model name, exact revision, generation commit, entity spans, harmonised spatial spans, exact-match scores and telemetry. It is a reliability asset for this notebook, **not** the frozen held-out benchmark result.

Set `FAST_MODE = False` to download the pinned model and reproduce the comparison live.

In [ ]:
RUN_HF = not FAST_MODE
hf_results = []
hf_fallback_path = repo_dir / "demo" / "ner_transformer_teaching_fallback_v1.json"

if RUN_HF:
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-transformers.txt"],check=True)
    from spatio_textual.transformer_ner import HFNERAnnotator
    hf = HFNERAnnotator("dslim/bert-base-NER", revision="0b95561fd0c304538b5eb8a0ee532ca24dd009b9", link_places=False)
    for rec in records:
        out = hf.annotate(rec["text"])
        pred = harmonize_ner_entities(out["entities"])
        ref = reference_spans_for_ner(rec)
        score = score_span_annotations(pred, ref, match="exact")
        hf_results.append({
            "example_id": rec["example_id"],
            "method": f"HF {hf.model_identifier}",
            "precision": score["precision"],
            "recall": score["recall"],
            "f1": score["f1"],
            "latency_ms": out["telemetry"][0]["latency_ms"],
            "error": out.get("error"),
        })
    hf_source = "live revision-pinned Hugging Face run"
else:
    fallback = json.loads(hf_fallback_path.read_text(encoding="utf-8"))
    assert [row["example_id"] for row in fallback["records"]] == [rec["example_id"] for rec in records]
    hf_results = [
        {
            "example_id": row["example_id"],
            "method": f"HF {fallback['model']}@{fallback['revision'][:8]} (precomputed)",
            "precision": row["exact_score"]["precision"],
            "recall": row["exact_score"]["recall"],
            "f1": row["exact_score"]["f1"],
            "latency_ms": (row.get("telemetry") or {}).get("latency_ms"),
            "error": None,
        }
        for row in fallback["records"]
    ]
    hf_source = "precomputed revision-pinned Hugging Face fallback"

display(pd.DataFrame(hf_results))
print("HF source:", hf_source)
if not RUN_HF:
    print("Pinned model:", fallback["model"], "revision:", fallback["revision"])

## 8. Save contextual comparison evidence

The saved files are teaching/development evidence. They are not yet the final held-out keynote result.

In [ ]:
out_dir=repo_dir/"sh2026_outputs"/"comparisons"
out_dir.mkdir(parents=True,exist_ok=True)
main_path=out_dir/"contextual_ner_teaching_reference.csv"
comparison_df.assign(evidence_status="NOT_A_BENCHMARK: visible development set").to_csv(main_path,index=False)
head_path=out_dir/"contextual_ner_toponym_head_to_head.csv"
pd.DataFrame(head_to_head).to_csv(head_path,index=False)
print("Saved:",main_path)
print("Saved:",head_path)
if hf_results:
    hf_path=out_dir/"hf_ner_teaching_reference.csv"
    pd.DataFrame(hf_results).to_csv(hf_path,index=False)
    print("Saved:",hf_path)
print("NOT A BENCHMARK: visible development set; do not use these rows to rank methods.")


## 9. What changed when we moved from rules to contextual NER?

| Dimension | Rules/gazetteer | Contextual NER |
|---|---|---|
| Vocabulary | explicitly supplied | learned from training data |
| Unseen forms | often brittle | can generalise contextually |
| Native ontology | ours to define | inherited from training task |
| Reproducibility | highly deterministic | normally stable at inference, but model/version dependent |
| Why a span appeared | inspect a rule/resource | harder to explain causally |
| Domain extension | edit resources/rules | retrain/fine-tune or add hybrid rules |
| Implicit relations | limited unless encoded | still outside standard NER |

The move to contextual NLP changes the balance; it does not eliminate the earlier methods.

## 10. Keynote evidence emerging already

This notebook gives us three distinct quantities worth visualising later:

1. **NER accuracy within its intended ontology**;
2. **ontology ceiling / representational reach**;
3. **cost/latency and inspectability of adding domain rules**.

This prevents a misleading slide where one single F1 number is used to rank fundamentally different tasks.

### Core message

> **Context buys generalisation, but the model still sees the world through the labels it was trained to predict. Spatial Humanities often needs a richer ontology than named entities alone.**

Next: **04 - Linking, ambiguity and historical geography**, where we separate textual recognition from claims about real-world places.